In [1]:
# Common imports
import numpy as np
import os
import tarfile
from six.moves import urllib # NB: 'six' provides Python 2/3 - agnostic versions of common python libraries
import pandas as pd

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

# Ignore useless warnings (see SciPy issue #5998)
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")

## 1. NASA FIRMS - Data Loading & Initial Exploration

The NASA FIRMS dataset contains satellite observations of thermal anomalies, including location, acquisition date, and Fire Radiative Power (FRP).

The initial exploration aims to understand the dataset structure, identify potential data quality issues, and determine which observations are relevant for wildfire intensity prediction.

In [2]:
# Load NASA FIRMS dataset
firms = pd.read_csv("fire_archive_SV-C2_851256.csv")

print("FIRMS shape:", firms.shape)
display(firms.head())

FIRMS shape: (2099465, 15)


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,40.63593,-74.22243,304.95,0.44,0.38,2023-01-01,639,SNPP,SNPP,n,2,284.65,1.89,N,2
1,40.63527,-74.21728,300.21,0.44,0.38,2023-01-01,639,SNPP,SNPP,n,2,282.67,1.41,N,2
2,39.96399,-76.59748,299.41,0.53,0.42,2023-01-01,640,SNPP,SNPP,n,2,279.76,1.05,N,2
3,37.41987,-77.43283,327.20,0.54,0.42,2023-01-01,640,SNPP,SNPP,n,2,283.63,3.07,N,0
4,38.73565,-77.66982,313.24,0.38,0.43,2023-01-01,640,SNPP,SNPP,n,2,281.92,1.38,N,0


In [3]:
# Inspect NASA FIRMS dataset
firms.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2099465 entries, 0 to 2099464
Data columns (total 15 columns):
 #   Column      Dtype  
---  ------      -----  
 0   latitude    float64
 1   longitude   float64
 2   brightness  float64
 3   scan        float64
 4   track       float64
 5   acq_date    object 
 6   acq_time    int64  
 7   satellite   object 
 8   instrument  object 
 9   confidence  object 
 10  version     int64  
 11  bright_t31  float64
 12  frp         float64
 13  daynight    object 
 14  type        int64  
dtypes: float64(7), int64(3), object(5)
memory usage: 240.3+ MB


In [4]:
# Summary statistics
firms.describe()

,latitude,longitude,brightness,scan,track,acq_time,version,bright_t31,frp,type
count,2.099465e+06,2.099465e+06,2.099465e+06,2.099465e+06,2.099465e+06,2.099465e+06,2099465.0,2.099465e+06,2.099465e+06,2.099465e+06
mean,5.311825e+01,-1.080615e+02,3.272029e+02,4.585731e-01,4.748121e-01,1.398902e+03,2.0,2.909914e+02,1.405299e+01,1.417290e-01
std,1.001127e+01,1.629275e+01,2.163145e+01,8.731980e-02,1.173249e-01,5.310167e+02,0.0,1.207730e+01,3.702236e+01,5.666379e-01
min,2.400046e+01,-1.300000e+02,2.080000e+02,3.200000e-01,3.600000e-01,4.470000e+02,2.0,0.000000e+00,0.000000e+00,0.000000e+00
25%,5.043847e+01,-1.200306e+02,3.069000e+02,3.900000e-01,3.700000e-01,9.320000e+02,2.0,2.837000e+02,1.780000e+00,0.000000e+00
50%,5.635700e+01,-1.138915e+02,3.291500e+02,4.300000e-01,4.300000e-01,1.111000e+03,2.0,2.891800e+02,4.100000e+00,0.000000e+00
75%,5.979135e+01,-9.926206e+01,3.435800e+02,5.100000e-01,5.600000e-01,1.944000e+03,2.0,2.970900e+02,1.066000e+01,0.000000e+00
max,7.164606e+01,-6.004692e+01,3.670000e+02,8.000000e-01,7.800000e-01,2.308000e+03,2.0,3.869300e+02,4.191100e+03,3.000000e+00


In [5]:
# Check missing values and duplicates
print("Missing values:")
print(firms.isnull().sum())

print("\nDuplicate rows:", firms.duplicated().sum())

Missing values:
latitude      0
longitude     0
brightness    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_t31    0
frp           0
daynight      0
type          0
dtype: int64

Duplicate rows: 0


In [6]:
# Convert acquisition date to datetime
firms["acq_date"] = pd.to_datetime(firms["acq_date"])

# Check date range and geographic coordinates
print("Date range:", firms["acq_date"].min(), "to", firms["acq_date"].max())

print("\nLatitude range:", firms["latitude"].min(), "to", firms["latitude"].max())
print("Longitude range:", firms["longitude"].min(), "to", firms["longitude"].max())

Date range: 2023-01-01 00:00:00 to 2023-12-31 00:00:00

Latitude range: 24.00046 to 71.64606
Longitude range: -130.0 to -60.04692


In [7]:
# Check fire detection types
print(firms["type"].value_counts().sort_index())

print("\nConfidence levels:")
print(firms["confidence"].value_counts())

type
0    1970880
2      88200
3      40385
Name: count, dtype: int64

Confidence levels:
confidence
n    1888588
h     121815
l      89062
Name: count, dtype: int64


### NASA FIRMS: Dataset Description & Initial Findings

**Dataset Variables**

| Variable | Description |
|---|---|
| 'latitude' | Latitude of the detection |
| 'longitude' | Longitude of the detection |
| 'brightness' | Brightness temperature (K) |
| 'scan' | Pixel dimension across the scan (km) |
| 'track' | Pixel dimension along the track (km) |
| 'acq_date' | Satellite acquisition date |
| 'acq_time' | Acquisition time (UTC, HHMM) |
| 'satellite' | Satellite identifier |
| 'instrument' | Satellite instrument |
| 'confidence' | Detection confidence (low, nominal, high) |
| 'version' | Data processing version |
| 'bright_t31' | Thermal brightness temperature (K) |
| 'frp' | Fire Radiative Power (MW) |
| 'daynight' | Day or night detection |
| 'type' | Presumed source of thermal anomaly |

**Fire Detection Types**

According to NASA FIRMS documentation:

- 0: Presumed vegetation fire
- 1: Active volcano
- 2: Other static land source
- 3: Offshore detection

**Initial Findings**

- The dataset contains **2,099,465 observations and 15 variables**, covering January–December 2023.
- No missing values or duplicate rows were detected.
- Geographic coordinates are within the requested study area (covering much of Canada and the continental United States)
- FRP ranges from 0 to 4,191.1 MW, with a median of 4.1 MW, indicating a highly right-skewed distribution.
- Most detections (1,970,880) are classified as presumed vegetation fires (type = 0).
- The 'confidence' attribute provides detection reliability information rather than fire intensity.

**Preprocessing Considerations**

For the initial wildfire analysis, we plan to retain 'type = 0'. However, this category includes presumed vegetation fires, which may also be agricultural or grassland fires.

Extreme FRP values should be investigated rather than automatically removed, as they may represent genuine high-intensity fire activity.

**Source:** [NASA FIRMS — VIIRS Active Fire Data](https://firms.modaps.eosdis.nasa.gov/content/descriptions/FIRMS_VIIRS_Firehotspots.html)


## 2. NASA POWER - Data Loading & Initial Exploration

In [8]:
# Load NASA POWER weather dataset
power = pd.read_csv(
    "POWER_Point_Daily_20230101_20231231_054d00N_0115d00W_UTC.csv",
    skiprows=15,
    na_values=-999
)

print("POWER shape:", power.shape)
display(power.head())
power.info()


POWER shape: (365, 9)


,YEAR,DOY,T2M,T2M_MAX,RH2M,WS10M,WD10M,PRECTOTCORR,T2MDEW
0,2023,1,-8.61,-2.56,97.66,3.73,274.9,0.01,-9.32
1,2023,2,-9.12,-4.44,95.89,3.42,257.3,0.00,-10.12
2,2023,3,-9.43,-4.74,97.60,2.66,303.5,0.10,-10.19
3,2023,4,-11.42,-5.55,97.06,3.73,287.3,0.00,-12.27
4,2023,5,-13.11,-9.04,98.13,3.01,186.9,0.03,-13.51


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   YEAR         365 non-null    int64  
 1   DOY          365 non-null    int64  
 2   T2M          365 non-null    float64
 3   T2M_MAX      365 non-null    float64
 4   RH2M         365 non-null    float64
 5   WS10M        365 non-null    float64
 6   WD10M        365 non-null    float64
 7   PRECTOTCORR  365 non-null    float64
 8   T2MDEW       365 non-null    float64
dtypes: float64(7), int64(2)
memory usage: 25.8 KB


In [9]:
# Check summary statistics
display(power.describe())

# Check missing values and duplicates
print("Missing values:")
print(power.isna().sum())

print("\nDuplicate rows:", power.duplicated().sum())

,YEAR,DOY,T2M,T2M_MAX,RH2M,WS10M,WD10M,PRECTOTCORR,T2MDEW
count,365.0,365.000000,365.000000,365.000000,365.000000,365.000000,365.000000,365.000000,365.000000
mean,2023.0,183.000000,5.067973,11.046274,77.842164,3.202822,214.739178,1.491096,0.790795
std,0.0,105.510663,10.662525,11.758883,12.440130,1.157195,89.989421,4.753759,9.115244
min,2023.0,1.000000,-23.180000,-20.260000,46.000000,0.860000,0.700000,0.000000,-26.080000
25%,2023.0,92.000000,-2.950000,1.230000,69.340000,2.410000,139.900000,0.020000,-5.810000
50%,2023.0,183.000000,5.220000,12.120000,77.470000,3.080000,251.000000,0.170000,-0.040000
75%,2023.0,274.000000,14.780000,21.940000,87.900000,3.780000,283.400000,0.780000,8.380000
max,2023.0,365.000000,22.300000,30.530000,99.880000,8.640000,359.500000,56.900000,16.040000


Missing values:
YEAR           0
DOY            0
T2M            0
T2M_MAX        0
RH2M           0
WS10M          0
WD10M          0
PRECTOTCORR    0
T2MDEW         0
dtype: int64

Duplicate rows: 0


In [10]:
# Convert year and day of year to date
power["date"] = pd.to_datetime(
    power["YEAR"].astype(str) + power["DOY"].astype(str).str.zfill(3),
    format="%Y%j"
)

print("Date range:", power["date"].min(), "to", power["date"].max())
display(power[["YEAR", "DOY", "date"]].head())


Date range: 2023-01-01 00:00:00 to 2023-12-31 00:00:00


,YEAR,DOY,date
0,2023,1,2023-01-01
1,2023,2,2023-01-02
2,2023,3,2023-01-03
3,2023,4,2023-01-04
4,2023,5,2023-01-05


### NASA POWER: Dataset Description & Initial Findings

NASA POWER (Prediction Of Worldwide Energy Resources) provides daily weather and atmospheric data.

**Dataset Variables**

| Variable | Description |
|---|---|
| 'YEAR' | Observation year |
| 'DOY' | Day of Year (1–365 or 366) |
| 'T2M' | Mean air temperature at 2 meters (°C) |
| 'T2M_MAX' | Maximum air temperature at 2 meters (°C) |
| 'RH2M' | Relative humidity at 2 meters (%) |
| 'WS10M' | Mean wind speed at 10 meters (m/s) |
| 'WD10M' | Wind direction at 10 meters (degrees) |
| 'PRECTOTCORR' | Corrected daily precipitation (mm/day) |
| 'T2MDEW' | Dew point temperature at 2 meters (°C) |

**Initial Findings**

- The dataset contains **365 daily observations and 9 original variables**, covering January-December 2023.
- Weather data correspond to a single location in Alberta, Canada (54°N, 115°W).
- No missing values or duplicate rows were detected.
- Temperature, humidity, wind, and precipitation variables are available throughout the year.
- A 'date' column was created from 'YEAR' and 'DOY' to support temporal alignment with NASA FIRMS.

**Methodological Considerations**

- NASA POWER provides historical weather conditions, not archived weather forecasts.
- Our objective is to predict wildfire intensity on day J+1 using information available on day J.
- Using actual weather observations from J+1 as input features would introduce **data leakage**.
- Archived weather forecasts issued on or before J would allow us to use predicted weather conditions for J+1 without leakage.
- The current POWER dataset covers only one location, while FIRMS covers a much larger region. Additional geographically matched weather data will be required.

**Next Step**

Investigate archived weather forecasts and their compatibility with NASA FIRMS before finalizing the data integration pipeline.

**Source:** [NASA POWER — Daily API Documentation](https://power.larc.nasa.gov/docs/services/api/temporal/daily/)



## 3. Data Engineering Strategy & Pipeline Design

### Scaling Strategy: Multiple Years and Locations

The initial analysis uses NASA FIRMS data for 2023 and NASA POWER weather data for one location in Alberta.

This is a prototype intended to validate data loading, cleaning, and preprocessing.

The final pipeline will be extended to multiple years and geographic grid cells using reusable Python functions.

**Key considerations:**
- Use a consistent spatial grid across all years.
- Aggregate FIRMS detections by grid cell and day.
- Retrieve historical weather and archived forecasts for corresponding locations.
- Ensure that weather forecasts were available before the prediction date.
- Validate temporal and spatial alignment before merging.

The final study period and grid resolution will be selected after verifying archived forecast availability.


**Data Ingestion Strategy**

NASA FIRMS historical data can be retrieved through the Archive Download service or the FIRMS API.

For this project, bulk historical CSV files will initially be requested through NASA Archive Download. Once available, the files will be stored locally and automatically processed by the Python pipeline.

The FIRMS API remains an alternative for automated historical retrieval or future updates.



## 4. Data Cleaning - NASA FIRMS & POWER
### 4.1. NASA FIRMS

In [11]:
# Keep presumed vegetation fires only
firms_clean = firms[firms["type"] == 0].copy()

print("Original observations:", len(firms))
print("Filtered observations:", len(firms_clean))
print("Removed observations:", len(firms) - len(firms_clean))


Original observations: 2099465
Filtered observations: 1970880
Removed observations: 128585


In [12]:
# Validate FRP values
print("FRP = 0:", (firms_clean["frp"] == 0).sum())
print("FRP < 0:", (firms_clean["frp"] < 0).sum())

print("\nFRP distribution:")
display(firms_clean["frp"].describe())



FRP = 0: 2895
FRP < 0: 0

FRP distribution:


count    1.970880e+06
mean     1.478563e+01
std      3.807945e+01
min      0.000000e+00
25%      1.900000e+00
50%      4.420000e+00
75%      1.141000e+01
max      4.191100e+03
Name: frp, dtype: float64

In [13]:
# Examine zero-FRP detections
zero_frp = firms_clean[firms_clean["frp"] == 0]

print("Confidence levels for zero-FRP detections:")
print(zero_frp["confidence"].value_counts())

print("\nDay/Night distribution:")
print(zero_frp["daynight"].value_counts())


Confidence levels for zero-FRP detections:
confidence
l    1778
h     693
n     424
Name: count, dtype: int64

Day/Night distribution:
daynight
D    2847
N      48
Name: count, dtype: int64


**Zero-FRP Observations - Findings**

Among the 2,895 observations with 'FRP = 0':

- 61.4% have low confidence.
- 23.9% have high confidence.
- 98.3% were detected during daytime.

Most zero-FRP observations are associated with low confidence and daytime detections. However, 693 observations have high confidence, so zero FRP values cannot automatically be considered invalid.

**Decision:** Retain these observations for now and document this limitation before further preprocessing.


In [14]:
# Check invalid geographic coordinates
print("Invalid coordinates:",
      (~firms_clean["latitude"].between(-90, 90) |
       ~firms_clean["longitude"].between(-180, 180)).sum())

# Check negative brightness temperatures (Kelvin)
print("Negative brightness:",
      (firms_clean["brightness"] < 0).sum())

print("Negative bright_t31:",
      (firms_clean["bright_t31"] < 0).sum())


Invalid coordinates: 0
Negative brightness: 0
Negative bright_t31: 0


## 4.2. NASA POWER

In [15]:
# Create a copy of NASA POWER dataset
power_clean = power.copy()

# Validate weather measurements
print("Invalid humidity:", (~power_clean["RH2M"].between(0, 100)).sum())
print("Negative wind speed:", (power_clean["WS10M"] < 0).sum())
print("Invalid wind direction:", (~power_clean["WD10M"].between(0, 360)).sum())
print("Negative precipitation:", (power_clean["PRECTOTCORR"] < 0).sum())


Invalid humidity: 0
Negative wind speed: 0
Invalid wind direction: 0
Negative precipitation: 0


In [16]:
# Final validation
print("FIRMS observations:", len(firms_clean))
print("POWER observations:", len(power_clean))

print("\nFIRMS missing values:", firms_clean.isna().sum().sum())
print("POWER missing values:", power_clean.isna().sum().sum())

FIRMS observations: 1970880
POWER observations: 365

FIRMS missing values: 0
POWER missing values: 0


In [17]:
# Final duplicate checks
print("FIRMS duplicate rows:", firms_clean.duplicated().sum())
print("POWER duplicate rows:", power_clean.duplicated().sum())
print("POWER duplicate dates:", power_clean["date"].duplicated().sum())


FIRMS duplicate rows: 0
POWER duplicate rows: 0
POWER duplicate dates: 0


### Data Cleaning: Findings & Decisions

**NASA FIRMS**
- Retained 1,970,880 presumed vegetation fire detections (type = 0).
- No missing values or invalid geographic coordinates were detected.
- No negative FRP or brightness temperatures were found.
- Retained 2,895 zero-FRP observations, as they are not necessarily invalid.
- Preserved all confidence levels (low, nominal, high).

**NASA POWER**
- Retained all 365 daily weather observations.
- No missing values, duplicate dates, or invalid weather measurements were detected.
- Created a date column for temporal alignment.

**Decision:** Both datasets are ready for the next processing stage. No additional records were removed beyond the FIRMS fire-type filtering.

**Limitation:** The POWER dataset currently covers only one geographic location. Additional weather data will be needed for spatial integration.


## 5. Spatial & Temporal Aggregation

NASA FIRMS provides individual satellite fire detections, each with its own geographic coordinates, detection date, and Fire Radiative Power (FRP).

NASA POWER provides historical weather observations for specific geographic locations. Our current weather dataset covers only one location in Alberta.

To combine wildfire and weather data, we need to organize both datasets by location and date.

**Spatial Aggregation**

For the initial prototype, we divide the study area into geographic cells of 0.5° × 0.5° and group fire detections within each cell.

We chose 0.5° because NASA POWER uses a similar geographic resolution for some weather variables, making it easier to combine fire and weather data.

This is a temporary choice. We may adjust the cell size after checking the geographic resolution of the historical weather forecast dataset.

**Temporal Aggregation**

We group fire detections by day to match the daily weather data.

For each geographic cell and day, we will calculate:
- Number of fire detections
- Maximum FRP (MW)
- Mean FRP (MW)
- Total FRP (MW)

**Next Step**

We will use these daily fire measurements to prepare the target for predicting next-day wildfire intensity.

Historical weather forecasts, rather than actual future weather observations, will be needed to build a realistic prediction model and avoid data leakage.

In [18]:
# Define spatial grid resolution
grid_size = 0.5

# Assign each detection to a geographic grid cell
firms_clean["grid_lat"] = (
    np.floor(firms_clean["latitude"] / grid_size) * grid_size
)

firms_clean["grid_lon"] = (
    np.floor(firms_clean["longitude"] / grid_size) * grid_size
)

display(firms_clean[
    ["latitude", "longitude", "grid_lat", "grid_lon"]
].head())


,latitude,longitude,grid_lat,grid_lon
3,37.41987,-77.43283,37.0,-77.5
4,38.73565,-77.66982,38.5,-78.0
5,37.36541,-83.26420,37.0,-83.5
7,34.78837,-79.40245,34.5,-79.5
8,34.79257,-79.40182,34.5,-79.5


In [19]:
# Aggregate fire detections by grid cell and day
firms_daily = (
    firms_clean.groupby(["grid_lat", "grid_lon", "acq_date"])
    .agg(
        fire_count=("frp", "size"),
        max_frp=("frp", "max"),
        mean_frp=("frp", "mean"),
        total_frp=("frp", "sum")
    )
    .reset_index()
)

print("Daily grid observations:", len(firms_daily))
display(firms_daily.head())


Daily grid observations: 119362


,grid_lat,grid_lon,acq_date,fire_count,max_frp,mean_frp,total_frp
0,24.0,-110.5,2023-03-21,1,2.21,2.21,2.21
1,24.0,-110.5,2023-06-14,1,6.81,6.81,6.81
2,24.0,-110.5,2023-06-27,1,3.04,3.04,3.04
3,24.0,-110.5,2023-06-30,1,2.58,2.58,2.58
4,24.0,-110.5,2023-07-18,2,1.92,1.63,3.26


In [20]:
# Validate daily aggregation
print("Original detections:", len(firms_clean))
print("Aggregated detections:", firms_daily["fire_count"].sum())

print("Duplicate grid-date combinations:",
      firms_daily.duplicated(["grid_lat", "grid_lon", "acq_date"]).sum())

print("Missing values:", firms_daily.isna().sum().sum())


Original detections: 1970880
Aggregated detections: 1970880
Duplicate grid-date combinations: 0
Missing values: 0


### Spatial & Temporal Aggregation - Findings

- Applied a provisional 0.5° × 0.5° geographic grid.
- Aggregated 1,970,880 FIRMS detections into 119,362 daily grid observations.
- Each row represents one geographic cell on one day.
- Calculated fire detection count, maximum FRP, mean FRP, and total FRP.
- Validation confirmed that all detections were accounted for, with no duplicate grid-date combinations or missing values.

**Important:** Fire detection count does not necessarily represent the number of individual wildfires.

**Limitation:** The dataset currently includes only grid cells and dates with at least one satellite fire detection. Days without detections are not represented.


## 6. Weather Data Preparation & Integration


In [21]:
# Check temporal coverage before integration
print("FIRMS period:",
      firms_daily["acq_date"].min(),
      "to", firms_daily["acq_date"].max())

print("POWER period:",
      power_clean["date"].min(),
      "to", power_clean["date"].max())


FIRMS period: 2023-01-01 00:00:00 to 2023-12-31 00:00:00
POWER period: 2023-01-01 00:00:00 to 2023-12-31 00:00:00


In [22]:
# Check FIRMS detections near the POWER location Alberta (54°N, −115°W)
near_power = firms_daily[
    firms_daily["grid_lat"].between(53.5, 54.5) &
    firms_daily["grid_lon"].between(-115.5, -114.5)
]

print("Daily fire observations near POWER location:", len(near_power))
display(near_power.head())


Daily fire observations near POWER location: 87


,grid_lat,grid_lon,acq_date,fire_count,max_frp,mean_frp,total_frp
101419,53.5,-115.5,2023-02-26,2,20.24,13.195000,26.39
101420,53.5,-115.5,2023-04-30,263,96.00,12.415437,3265.26
101421,53.5,-115.5,2023-05-01,83,29.93,4.158072,345.12
101422,53.5,-115.5,2023-05-02,16,5.45,2.872500,45.96
101423,53.5,-115.5,2023-05-03,7,4.85,2.971429,20.80


In [23]:
# Count daily observations by geographic cell
display(
    near_power.groupby(["grid_lat", "grid_lon"])
    .size()
    .reset_index(name="daily_observations")
)


,grid_lat,grid_lon,daily_observations
0,53.5,-115.5,21
1,53.5,-115.0,7
2,53.5,-114.5,16
3,54.0,-115.5,3
4,54.0,-115.0,16
5,54.0,-114.5,9
6,54.5,-115.5,9
7,54.5,-115.0,1
8,54.5,-114.5,5


In [24]:
# Select the FIRMS grid cell at the POWER reference coordinates
fire_local = firms_daily[
    (firms_daily["grid_lat"] == 54.0) &
    (firms_daily["grid_lon"] == -115.0)
].copy()

print("Daily fire observations:", len(fire_local))
display(fire_local.head())


Daily fire observations: 16


,grid_lat,grid_lon,acq_date,fire_count,max_frp,mean_frp,total_frp
102567,54.0,-115.0,2023-01-03,4,7.40,4.737500,18.95
102568,54.0,-115.0,2023-01-15,3,5.85,4.733333,14.20
102569,54.0,-115.0,2023-01-20,2,4.12,3.875000,7.75
102570,54.0,-115.0,2023-01-28,1,2.83,2.830000,2.83
102571,54.0,-115.0,2023-02-05,2,7.38,7.235000,14.47


In [25]:
# Merge fire detections with daily weather observations
fire_weather = fire_local.merge(
    power_clean,
    left_on="acq_date",
    right_on="date",
    how="left"
)

print("Merged observations:", len(fire_weather))
print("Missing weather values:", fire_weather["T2M"].isna().sum())

display(fire_weather[
    ["acq_date", "fire_count", "max_frp",
     "T2M", "RH2M", "WS10M", "PRECTOTCORR"]
].head())


Merged observations: 16
Missing weather values: 0


,acq_date,fire_count,max_frp,T2M,RH2M,WS10M,PRECTOTCORR
0,2023-01-03,4,7.40,-9.43,97.60,2.66,0.10
1,2023-01-15,3,5.85,-7.65,98.60,0.86,0.22
2,2023-01-20,2,4.12,-3.00,91.45,4.51,0.02
3,2023-01-28,1,2.83,-18.44,83.56,4.62,0.20
4,2023-02-05,2,7.38,-2.95,96.07,3.58,0.36


In [26]:
# Check consecutive fire detection days
fire_weather = fire_weather.sort_values("acq_date")

print("Consecutive days:",
      (fire_weather["acq_date"].diff().dt.days == 1).sum())

print("Total days with detections:", len(fire_weather))


Consecutive days: 5
Total days with detections: 16


In [27]:
# Validate the merged dataset
print("Duplicate grid-date combinations:",
      fire_weather.duplicated(
          ["grid_lat", "grid_lon", "acq_date"]
      ).sum())

print("Missing values:",
      fire_weather.isna().sum().sum())

print("Merged observations:", len(fire_weather))


Duplicate grid-date combinations: 0
Missing values: 0
Merged observations: 16


### Weather Data Integration: Findings & Decisions

**Findings**
- Both NASA FIRMS and NASA POWER cover January 1 to December 31, 2023.
- Identified 87 daily fire observations across 9 geographic grid cells near the POWER reference location (54°N, -115°W).
- Selected one FIRMS grid cell (54.0, -115.0) for the initial integration test.
- Successfully merged 16 daily fire observations with NASA POWER weather data using matching dates.
- Validation confirmed no missing values or duplicate grid-date combinations.
- Only 5 pairs of consecutive days with fire detections were identified.

**Decisions**
- Use daily data as the initial temporal resolution.
- Use historical weather observations to model fire intensity under given weather conditions.
- Retain the 0.5° × 0.5° geographic grid provisionally.

**Limitations**
- The integration currently covers only one geographic cell and one weather reference location.
- Spatial matching remains approximate and requires validation.
- Days without satellite fire detections are not represented.
- The 16 merged observations are sufficient for testing the integration process, but not for training a reliable ML model.

**Next Step**
Extend the integration to multiple geographic locations and years to build a larger dataset for Machine Learning.


## 7. Additional Data Sources: Canadian Fire Behaviour Factors

### Canadian Fire Behaviour Prediction System

![Canadian Fire Behaviour Prediction System](Canadian%20Fire%20Behaviour.jpg)

This figure from Natural Resources Canada shows the main factors that influence fire behaviour.

Our proposed ML model:

**FRP = f(Weather, Fuel Type, Slope, FWI)**

Source: [Natural Resources Canada - Canadian Fire Behaviour Prediction System](https://natural-resources.canada.ca/forests-forestry/wildland-fires/canada-fire-behaviour-prediction-system)



In [28]:
# Load FWI dataset
fwi = pd.read_csv("cwfis_fwi2020sv3.0_ll.csv")

print("FWI shape:", fwi.shape)
display(fwi.head())



FWI shape: (4230623, 23)


,rep_date,aes,wmo,name,lon,lat,elev,temp,td,rh,...,sog,ffmc,dmc,dc,isi,bui,fwi,dsr,opts,calcstatus
0,2020-01-01 12:00:00,1012475,71031,DISCOVERY+IS...,-123.23,48.42,19,9.5,8.6,94.0,...,0.0,8.9,0.1,85.6,0.0,0.1,0.0,0.0,IDW=R:M=1:,1
1,2020-01-01 12:00:00,1012710,71798,ESQUIMALT+HAR...,-123.43,48.43,3,10.2,6.2,76.0,...,0.0,35.6,0.3,1.5,0.1,0.4,0.0,0.0,M=1:,1
2,2020-01-01 12:00:00,1014820,71774,MAL...,-123.53,48.57,366,9.4,5.5,76.9,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDW=RH:WSD:R:M=1:,-3
3,2020-01-01 12:00:00,1015630,71927,NORTH+COWI...,-123.72,48.82,45,11.6,4.4,61.0,...,0.0,36.9,0.5,1.8,0.0,0.6,0.0,0.0,M=1:,1
4,2020-01-01 12:00:00,1016640,71778,RACE+R...,-123.53,48.30,7,9.2,6.9,85.0,...,0.0,28.8,0.2,1.4,0.0,0.2,0.0,0.0,IDW=R:M=1:,1


In [29]:
# Check dates and available records for 2023
fwi["rep_date"] = pd.to_datetime(fwi["rep_date"])

print("Date range:", fwi["rep_date"].min(), "to", fwi["rep_date"].max())
print("2023 observations:", (fwi["rep_date"].dt.year == 2023).sum())


Date range: 2020-01-01 12:00:00 to 2026-09-30 12:00:00
2023 observations: 614418


In [30]:
# Keep only 2023 observations
fwi_2023 = fwi[fwi["rep_date"].dt.year == 2023].copy()

print("FWI 2023 shape:", fwi_2023.shape)
print("Missing FWI values:", fwi_2023["fwi"].isna().sum())

display(fwi_2023.head())


FWI 2023 shape: (614418, 23)
Missing FWI values: 203702


,rep_date,aes,wmo,name,lon,lat,elev,temp,td,rh,...,sog,ffmc,dmc,dc,isi,bui,fwi,dsr,opts,calcstatus
1867078,2023-01-01 12:00:00,1012475,71031,DISCOVERY+IS...,-123.23,48.42,19,6.8,6.8,100.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDW=R:M=1:,-3
1867079,2023-01-01 12:00:00,1012710,71798,ESQUIMALT+HAR...,-123.43,48.43,3,6.3,4.0,86.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M=1:,-3
1867080,2023-01-01 12:00:00,1014820,71774,MAL...,-123.53,48.57,366,4.3,3.9,98.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDW=WSD:M=1:,-3
1867081,2023-01-01 12:00:00,1015630,71927,NORTH+COWI...,-123.72,48.82,45,4.0,3.1,94.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M=1:,-3
1867082,2023-01-01 12:00:00,1016640,71778,RACE+R...,-123.53,48.30,7,6.6,4.9,88.8,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDW=T:RH:R:M=1:,-3


In [31]:
# Check missing FWI values by month
display(
    fwi_2023.groupby(fwi_2023["rep_date"].dt.month)["fwi"]
    .agg(total="size", missing=lambda x: x.isna().sum())
)


,total,missing
rep_date,,
1,43118,35866
2,38938,32594
3,43191,35459
4,43163,25522
5,56864,8681
6,58495,3657
7,60412,1730
8,60246,1077
9,58271,1734


In [32]:
# Check FWI observations near our Alberta location
fwi_local = fwi_2023[
    fwi_2023["lat"].between(53, 55) &
    fwi_2023["lon"].between(-116, -114)
]

print("Nearby FWI observations:", len(fwi_local))
print("Available FWI values:", fwi_local["fwi"].notna().sum())

display(fwi_local[["rep_date", "name", "lat", "lon", "fwi"]].head())


Nearby FWI observations: 4738
Available FWI values: 3164


,rep_date,name,lat,lon,fwi
1867312,2023-01-01 12:00:00,BRETON+P...,53.08,-114.43,NaN
1867330,2023-01-01 12:00:00,GLENEVIS+...,53.83,-114.53,NaN
1867359,2023-01-01 12:00:00,TOMAHAWK+...,53.43,-114.72,NaN
1867368,2023-01-01 12:00:00,EDMONTON+STONY+PLAI...,53.55,-114.12,NaN
1867453,2023-01-01 12:00:00,BARRHEA...,54.10,-114.45,NaN


In [33]:
# Find nearby stations and their distance from POWER
stations = fwi_local[["name", "lat", "lon"]].drop_duplicates().copy()

stations["distance_km"] = (
    ((stations["lat"] - 54) * 111)**2 +
    ((stations["lon"] + 115) * 65)**2
)**0.5

display(stations.sort_values("distance_km").head(10))


,name,lat,lon,distance_km
1867454,CAMPSIE+...,54.13,-114.68,25.315310
1867330,GLENEVIS+...,53.83,-114.53,35.907929
1867453,BARRHEA...,54.10,-114.45,37.433575
1867457,EVANSBURG2+...,53.62,-115.07,42.424697
2024934,WHITECOURT AUTO,54.03,-115.72,46.918322
1867458,FORT+ASSINIBOINE+...,54.42,-114.77,48.958420
1867469,WHITECOU...,54.15,-115.78,53.363963
2016271,VEGA AUTO,54.43,-114.43,60.422309
2035433,IMPERIAL AUTO,54.47,-115.57,63.987588
1867359,TOMAHAWK+...,53.43,-114.72,65.835651


In [34]:
# Check FWI availability at the nearest station
nearest = stations.sort_values("distance_km").iloc[0]

station_data = fwi_local[
    (fwi_local["lat"] == nearest["lat"]) &
    (fwi_local["lon"] == nearest["lon"])
]

print("Station:", nearest["name"])
print("Total observations:", len(station_data))
print("Available FWI:", station_data["fwi"].notna().sum())

display(station_data[["rep_date", "fwi"]].dropna().head())


Station:                                       CAMPSIE+AUTO
Total observations: 365
Available FWI: 198


,rep_date,fwi
2003932,2023-04-09 12:00:00,12.9
2005339,2023-04-10 12:00:00,22.0
2006747,2023-04-11 12:00:00,12.2
2008154,2023-04-12 12:00:00,13.4
2009566,2023-04-13 12:00:00,13.9


In [35]:
# Prepare fire weather indices from CAMPSIE station
indices = ["ffmc", "isi", "bui", "fwi"]

station_fwi = station_data[["rep_date"] + indices].copy()
station_fwi["date"] = station_fwi["rep_date"].dt.normalize()

# Merge with FIRMS + POWER
fire_weather_fwi = fire_weather.merge(
    station_fwi[["date"] + indices],
    on="date",
    how="left",
    validate="one_to_one"
)

print("Merged observations:", len(fire_weather_fwi))
display(fire_weather_fwi[
    ["date", "max_frp", "ffmc", "isi", "bui", "fwi"]
].head(16))



Merged observations: 16


,date,max_frp,ffmc,isi,bui,fwi
0,2023-01-03,7.40,NaN,NaN,NaN,NaN
1,2023-01-15,5.85,NaN,NaN,NaN,NaN
2,2023-01-20,4.12,NaN,NaN,NaN,NaN
3,2023-01-28,2.83,NaN,NaN,NaN,NaN
4,2023-02-05,7.38,NaN,NaN,NaN,NaN
5,2023-02-06,1.26,NaN,NaN,NaN,NaN
6,2023-04-10,9.73,91.3,16.5,23.3,22.0
7,2023-06-26,6.59,87.5,6.4,32.1,12.7
8,2023-06-27,1.42,87.9,3.7,36.5,8.6
9,2023-06-28,15.93,87.9,4.2,40.7,10.2


### CWFIS Fire Weather Indices: Findings & Decisions

**Meaning of the Fields Used**

| Field | Meaning |
|---|---|
| 'rep_date | Weather observation date and time |
| 'name' | Weather station name |
| 'lat' | Latitude of the weather station |
| 'lon' | Longitude of the weather station |
| 'ffmc' | Fine Fuel Moisture Code: indicates how dry small surface fuels are |
| 'isi' | Initial Spread Index: indicates the potential rate of fire spread |
| 'bui' | Buildup Index: indicates the amount of fuel available for burning |
| 'fwi' | Fire Weather Index: indicates potential fire intensity based on weather conditions |

**Findings**

- Loaded 4,230,623 observations covering 2020 to September 2026.
- Selected 614,418 observations for 2023.
- Identified CAMPSIE AUTO as the nearest station to our Alberta reference point (approximately 25 km).
- Integrated four indices: FFMC, ISI, BUI, and FWI.
- Successfully merged CWFIS with the FIRMS + POWER prototype.
- Of the 16 merged observations, 6 have all four indices available.

**Decisions**

- Retain all 16 observations for now.
- Keep missing index values as NaN rather than replacing them with zero.
- Use the Canadian Fire Weather Indices as additional features for our ML model.

**Limitation**

The indices come from a nearby weather station, so geographic matching is approximate.

**Source:** [Canadian Wildland Fire Information System (CWFIS)](https://cwfis.cfs.nrcan.gc.ca/)



In [36]:
%pip install rasterio
import rasterio

with rasterio.open("mrdem-30-dtm.vrt") as src:
    print("Resolution:", src.res)
    print("CRS:", src.crs)
    print("Bands:", src.count)


Note: you may need to restart the kernel to use updated packages.
Resolution: (30.0, 30.0)
CRS: EPSG:3979
Bands: 1


In [37]:
from rasterio.warp import transform

with rasterio.open("mrdem-30-dtm.vrt") as src:
    x, y = transform("EPSG:4326", src.crs, [-115.0], [54.0])
    elevation = next(src.sample([(x[0], y[0])]))[0]

    print("Location: 54° N, -115° W")
    print("Elevation (m):", elevation)
    print("NoData value:", src.nodata)


Location: 54° N, -115° W
Elevation (m): 735.2396
NoData value: -32767.0


In [38]:
from rasterio.windows import Window

with rasterio.open("mrdem-30-dtm.vrt") as src:
    x, y = transform("EPSG:4326", src.crs, [-115.0], [54.0])
    row, col = src.index(x[0], y[0])

    # Read a 3x3 elevation window around the point
    elevations = src.read(
        1, window=Window(col - 1, row - 1, 3, 3)
    ).astype(float)

    if np.any(elevations == src.nodata):
        print("Missing elevation data in the neighborhood.")
    else:
        dz_dy, dz_dx = np.gradient(elevations, 30, 30)
        slope = np.degrees(np.arctan(
            np.sqrt(dz_dx[1, 1]**2 + dz_dy[1, 1]**2)
        ))

        print("Elevation (m):", elevations[1, 1])
        print("Slope (degrees):", round(slope, 2))


Elevation (m): 735.2396240234375
Slope (degrees): 3.95


In [39]:
# Add topographic features to the Alberta prototype
fire_weather_fwi["elevation"] = float(elevations[1, 1])
fire_weather_fwi["slope"] = float(slope)

# Display the main features
display(fire_weather_fwi[
    ["date", "max_frp", "ffmc", "isi", "bui", "fwi", "elevation", "slope"]
].head(16))


,date,max_frp,ffmc,isi,bui,fwi,elevation,slope
0,2023-01-03,7.40,NaN,NaN,NaN,NaN,735.239624,3.953987
1,2023-01-15,5.85,NaN,NaN,NaN,NaN,735.239624,3.953987
2,2023-01-20,4.12,NaN,NaN,NaN,NaN,735.239624,3.953987
3,2023-01-28,2.83,NaN,NaN,NaN,NaN,735.239624,3.953987
4,2023-02-05,7.38,NaN,NaN,NaN,NaN,735.239624,3.953987
5,2023-02-06,1.26,NaN,NaN,NaN,NaN,735.239624,3.953987
6,2023-04-10,9.73,91.3,16.5,23.3,22.0,735.239624,3.953987
7,2023-06-26,6.59,87.5,6.4,32.1,12.7,735.239624,3.953987
8,2023-06-27,1.42,87.9,3.7,36.5,8.6,735.239624,3.953987
9,2023-06-28,15.93,87.9,4.2,40.7,10.2,735.239624,3.953987


### Topography (MRDEM): Findings & Decisions

**Meaning of the Fields**

| Field | Meaning |
|---|---|
| 'elevation' | Terrain altitude above sea level, in meters |
| 'slope' | Terrain inclination, in degrees |

**Findings**

- Loaded the Canadian MRDEM Digital Terrain Model using Rasterio.
- Confirmed a spatial resolution of 30 meters (EPSG:3979).
- Extracted elevation and calculated slope at our Alberta reference point (54° N, -115° W).
- Elevation: 735.24 m.
- Slope: 3.95°.
- Successfully added both features to the 16-observation prototype.

**Decisions**

- Retain elevation and slope as potential ML features.
- Calculate location-specific topographic values when expanding the dataset.

**Limitation**

The current prototype uses the same elevation and slope values for all 16 observations. These values represent one reference point, not the entire grid cell.

**Source:** [Natural Resources Canada — MRDEM](https://open.canada.ca/data/en/dataset/18752265-bda3-498c-a4ba-9dfe68cb98da)


In [40]:
from pathlib import Path

fuel_folder = Path("National_FBP_Fueltypes_version2014b")

for file in fuel_folder.rglob("*"):
    if file.is_file():
        print(file.relative_to(fuel_folder), "—", round(file.stat().st_size / 1024**2, 2), "MB")


CIFFC_standard_colours_for_FBPfueltypes.pdf — 0.18 MB
ciffc_standard_fuels_legend.lyr — 0.01 MB
Generic_EN End-User Agreement for Digital Data.pdf — 0.11 MB
Generic_FR_End-User Agreement for Digital Data.pdf — 0.2 MB
nat_fbpfuels_2014b.tfw — 0.0 MB
nat_fbpfuels_2014b.tif — 375.56 MB
nat_fbpfuels_2014b.tif.aux.xml — 0.0 MB
nat_fbpfuels_2014b.tif.ovr — 12.0 MB
nat_fbpfuels_2014b.tif.vat.dbf — 0.0 MB
nat_fbpfuels_2014b.tif.xml — 0.0 MB
nat_fbpfuels_2014b_metadata.pdf — 0.52 MB
nat_fbpfuels_2014b_metadata2.pdf — 0.34 MB
info\arc.dir — 0.0 MB


In [41]:
fuel_path = "National_FBP_Fueltypes_version2014b/nat_fbpfuels_2014b.tif"

with rasterio.open(fuel_path) as src:
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Bands:", src.count)
    print("NoData:", src.nodata)
    print("Data type:", src.dtypes[0])


CRS: PROJCS["Canada_Lambert_Conformal_Conic",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",49],PARAMETER["central_meridian",-95],PARAMETER["standard_parallel_1",49],PARAMETER["standard_parallel_2",77],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Resolution: (250.0, 250.0)
Bands: 1
NoData: 0.0
Data type: uint8


In [42]:
from rasterio.warp import transform

with rasterio.open(fuel_path) as src:
    x, y = transform("EPSG:4326", src.crs, [-115.0], [54.0])
    fuel_type = int(next(src.sample([(x[0], y[0])]))[0])

    print("Location: 54° N, -115° W")
    print("Fuel type code:", fuel_type)
    print("NoData:", src.nodata)


Location: 54° N, -115° W
Fuel type code: 108
NoData: 0.0


In [43]:
# Add fuel type to the Alberta prototype
fire_weather_fwi["fuel_type"] = fuel_type
fire_weather_fwi["fuel_class"] = "D1"

display(fire_weather_fwi[
    ["date", "max_frp", "elevation", "slope", "fuel_type", "fuel_class"]
].head())


,date,max_frp,elevation,slope,fuel_type,fuel_class
0,2023-01-03,7.40,735.239624,3.953987,108,D1
1,2023-01-15,5.85,735.239624,3.953987,108,D1
2,2023-01-20,4.12,735.239624,3.953987,108,D1
3,2023-01-28,2.83,735.239624,3.953987,108,D1
4,2023-02-05,7.38,735.239624,3.953987,108,D1


### FBP Fuel Types: Findings & Decisions

**Meaning of the Fields**

| Field | Meaning |
|---|---|
| 'fuel_type' | Numeric code identifying the vegetation fuel type |
| 'fuel_class' | Canadian FBP fuel classification associated with the numeric code |

**Findings**

- Downloaded the Canadian National FBP Fuel Types 2014b dataset.
- Loaded the GeoTIFF file using Rasterio.
- Confirmed a spatial resolution of 250 meters.
- Extracted the fuel type at our Alberta reference point (54° N, -115° W).
- Fuel type code: 108 (D1 — Leafless Aspen).
- Successfully added `fuel_type` and `fuel_class` to the 16-observation prototype.

**Decisions**

- Retain fuel type as a categorical feature for the ML model.
- Extract location-specific fuel types when expanding the dataset.
- Use the 2014b map as a historical reference, subject to temporal validation.

**Limitations**

- The current prototype assigns the same fuel type to all 16 observations.
- A 0.5° grid cell may contain multiple fuel types.
- The 2014b map does not capture vegetation changes across the full 2012–2026 study period.

**Source:** [Canadian Wildland Fire Information System — Fuel Types](https://cwfis.cfs.nrcan.gc.ca/downloads/fuels/)


## 8. Reproducible Data Engineering Pipeline

### 8.1. Prototype Dataset Validation


In [44]:
# Final prototype validation
print("Dataset shape:", fire_weather_fwi.shape)
print("Date range:", fire_weather_fwi["date"].min(),
      "to", fire_weather_fwi["date"].max())

print("\nColumns:")
print(fire_weather_fwi.columns.tolist())

print("\nMissing values:")
print(fire_weather_fwi.isna().sum())

print("\nDuplicate grid-date records:",
      fire_weather_fwi.duplicated(
          subset=["grid_lat", "grid_lon", "date"]
      ).sum())


Dataset shape: (16, 25)
Date range: 2023-01-03 00:00:00 to 2023-12-04 00:00:00

Columns:
['grid_lat', 'grid_lon', 'acq_date', 'fire_count', 'max_frp', 'mean_frp', 'total_frp', 'YEAR', 'DOY', 'T2M', 'T2M_MAX', 'RH2M', 'WS10M', 'WD10M', 'PRECTOTCORR', 'T2MDEW', 'date', 'ffmc', 'isi', 'bui', 'fwi', 'elevation', 'slope', 'fuel_type', 'fuel_class']

Missing values:
grid_lat        0
grid_lon        0
acq_date        0
fire_count      0
max_frp         0
mean_frp        0
total_frp       0
YEAR            0
DOY             0
T2M             0
T2M_MAX         0
RH2M            0
WS10M           0
WD10M           0
PRECTOTCORR     0
T2MDEW          0
date            0
ffmc           10
isi            10
bui            10
fwi            10
elevation       0
slope           0
fuel_type       0
fuel_class      0
dtype: int64

Duplicate grid-date records: 0


### Prototype Dataset Validation: Findings

- The integrated prototype contains 16 observations and 25 columns.
- Observations cover January 3 to December 4, 2023.
- No duplicate grid-date records were found.
- NASA POWER weather variables are complete.
- Four CWFIS indices (FFMC, ISI, BUI, FWI) each have 10 missing values, mainly outside the fire-weather calculation season.
- Elevation, slope, and fuel type are available but constant across the prototype.

**Decision:** Retain the prototype as a validation of the five-source integration process. Expand the geographic and temporal coverage before ML training.

**Next step:** Develop a reproducible pipeline for the 2012–2026 study period, with location-specific weather, fire-weather indices, terrain, and fuel type.


### 8.2. Pipeline Configuration

Define the study period, spatial resolution, prediction target, and input features for the reproducible data engineering pipeline.


In [45]:
# Study period
START_DATE = "2012-01-20"
END_DATE = "2026-11-17"

# Spatial and temporal aggregation
GRID_SIZE = 0.5  # Degrees
TIME_FREQUENCY = "daily"

# Prediction target
TARGET = "max_frp"

# Data sources
DATA_SOURCES = {
    "fire": "NASA FIRMS - VIIRS S-NPP",
    "weather": "NASA POWER",
    "fire_weather": "CWFIS",
    "topography": "MRDEM",
    "fuel": "FBP Fuel Types 2014b"
}

# Planned input features
FEATURES = [
    "T2M", "RH2M", "WS10M", "PRECTOTCORR",
    "ffmc", "isi", "bui", "fwi",
    "elevation", "slope", "fuel_type"
]

print("Study period:", START_DATE, "to", END_DATE)
print("Grid size:", GRID_SIZE, "degrees")
print("Target:", TARGET)
print("Number of features:", len(FEATURES))
print("Data sources:", len(DATA_SOURCES))


Study period: 2012-01-20 to 2026-11-17
Grid size: 0.5 degrees
Target: max_frp
Number of features: 11
Data sources: 5


### 8.3. Data Collection Pipeline

#### 8.3.1. NASA FIRMS: Historical Fire Data

Identify the available historical FIRMS files before implementing automated loading and processing.


In [46]:
from pathlib import Path

firms_files = sorted(Path(".").glob("fire_archive*.csv"))

print("Available FIRMS files:", len(firms_files))

for file in firms_files:
    print(file.name, "-", round(file.stat().st_size / 1024**2, 2), "MB")


Available FIRMS files: 1
fire_archive_SV-C2_851256.csv - 164.29 MB


#### 8.3.2. Historical Data Coverage

Prepare the NASA FIRMS data collection for the 2012–2026 study period using VIIRS S-NPP observations.


In [47]:
available_years = set()

for file in firms_files:
    dates = pd.read_csv(file, usecols=["acq_date"])
    available_years.update(pd.to_datetime(dates["acq_date"]).dt.year.unique())

expected_years = set(range(2012, 2027))

print("Available years:", sorted(available_years))
print("Missing years:", sorted(expected_years - available_years))


Available years: [np.int32(2023)]
Missing years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2024, 2025, 2026]
